In [ ]:
from collections import Counter

# FP Tree Node
class Node:
    def __init__(self, item, count=1, parent=None):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None

In [ ]:
# Build FP Tree
def build_tree(transactions, min_sup):
    count = Counter(i for t in transactions for i in set(t))
    count = {i: c for i, c in count.items() if c >= min_sup}

    root = Node("ROOT")
    header = {i: None for i in count}

    for t in transactions:
        items = sorted(
            [i for i in t if i in count],
            key=lambda x: -count[x]
        )

        current = root

        for item in items:
            if item not in current.children:
                current.children[item] = Node(
                    item, 1, current
                )

                # Header link
                if header[item] is None:
                    header[item] = current.children[item]
                else:
                    p = header[item]
                    while p.link:
                        p = p.link
                    p.link = current.children[item]

            else:
                current.children[item].count += 1

            current = current.children[item]

    return root, header, count

In [ ]:
# Find conditional pattern base
def pattern_base(header, item):
    paths = []
    node = header[item]

    while node:
        path = []
        p = node.parent

        while p and p.item != "ROOT":
            path.append(p.item)
            p = p.parent

        if path:
            paths.append(path[::-1])

        node = node.link

    return paths

In [ ]:
# Recursive FP-Growth
def fp_growth(transactions, min_sup, prefix=[]):
    root, header, count = build_tree(transactions, min_sup)

    for item in sorted(count, key=count.get):
        new_pattern = prefix + [item]

        print(new_pattern, "Support =", count[item])

        base = pattern_base(header, item)

        if base:
            fp_growth(base, min_sup, new_pattern)

In [ ]:

# Transactions
transactions = [
    ["Bread", "Milk"],
    ["Bread", "Diaper", "Beer"],
    ["Milk", "Diaper", "Beer"],
    ["Bread", "Milk", "Diaper", "Beer"],
    ["Bread", "Milk", "Diaper"]
]

# Run FP-Growth
fp_growth(transactions, 2)

['Beer'] Support = 3
['Beer', 'Bread'] Support = 2
['Beer', 'Milk'] Support = 2
['Beer', 'Milk', 'Diaper'] Support = 2
['Beer', 'Diaper'] Support = 3
['Bread'] Support = 4
['Milk'] Support = 4
['Diaper'] Support = 4
['Diaper', 'Bread'] Support = 2
['Diaper', 'Milk'] Support = 2
